# Plain Python vs PyRust Engine

This notebook runs the **same functions** two ways, plain Python and PyRust (`optimize(...)`), checks that both give **exactly the same result**, and prints the time for each.

It covers what PyRust 0.6 can speed up: integer and float maths, `math` functions, helper functions, recursion, bit operations, reading and writing NumPy arrays, and what happens with code it can't speed up.

**You need Rust (`rustc`).** Without it everything still runs, but as plain Python (about 1×). Step 1 checks for it.

Run the cells from top to bottom (**Run → Run All Cells**).

## Step 1: Setup

In [ ]:
import os, shutil, subprocess, sys

# Jupyter often starts with a shorter PATH than your terminal; add the usual Rust locations.
for folder in [os.path.expanduser("~/.cargo/bin"), "/opt/homebrew/opt/rustup/bin", "/usr/local/opt/rustup/bin"]:
    if os.path.isdir(folder) and folder not in os.environ["PATH"].split(os.pathsep):
        os.environ["PATH"] = folder + os.pathsep + os.environ["PATH"]

print("Python:", sys.version.split()[0])
try:
    print("Rust:  ", subprocess.check_output([shutil.which("rustc") or "rustc", "--version"], text=True).strip())
    RUST_OK = True
except Exception:
    RUST_OK = False
    print("Rust:   NOT WORKING - PyRust will run everything as plain Python (no speedup).")
    print("        Install:  brew install rustup && rustup default stable   (or https://rustup.rs)")
    print("        Repair a broken install ('missing manifest'):")
    print("                  rustup toolchain uninstall stable && rustup default stable")
    print("        Then restart this notebook's kernel.")

In [ ]:
# Install PyRust Engine + NumPy into this notebook's Python.
# If you opened this notebook from a clone of the repository and ran `pip install -e ".[arrays]"`, skip this cell.
%pip install -q "pyrust-engine[arrays] @ git+https://github.com/surajmhaske02/pyrust-engine.git"

In [ ]:
import math, statistics, time, warnings
import numpy as np
import pyrust_engine
from pyrust_engine import optimize, F64Array, I64Array, PyRustFallbackWarning

print("PyRust Engine", pyrust_engine.__version__)

## Step 2: Timing helpers

`compare` runs both versions, checks the answers match (including any arrays they write), then prints the median time of several runs and the speedup.

In [ ]:
RESULTS = []

def measure(fn, make_args, repeat=7, number=1):
    samples = []
    for _ in range(repeat):
        args = make_args()
        start = time.perf_counter()
        for _ in range(number):
            fn(*args)
        samples.append((time.perf_counter() - start) / number)
    return statistics.median(samples)

def show(seconds):
    if seconds >= 1:    return f"{seconds:.3f} s"
    if seconds >= 1e-3: return f"{seconds * 1e3:.3f} ms"
    return f"{seconds * 1e6:.3f} µs"

def compare(name, python_fn, make_args, number=1):
    fast_fn = optimize(python_fn)
    start = time.perf_counter()
    fast_args = make_args()
    fast_result = fast_fn(*fast_args)          # first call compiles to Rust (or loads the disk cache)
    first_call = time.perf_counter() - start

    slow_args = make_args()
    slow_result = python_fn(*slow_args)
    assert repr(fast_result) == repr(slow_result), (fast_result, slow_result)
    for a, b in zip(fast_args, slow_args):
        if isinstance(a, np.ndarray):
            assert a.tobytes() == b.tobytes(), "arrays differ"

    t_python = measure(python_fn, make_args, number=number)
    t_pyrust = measure(fast_fn, make_args, number=number)
    speedup = t_python / t_pyrust
    backend = fast_fn.report()["backend"]
    print(f"{name}")
    print(f"  result (same in both): {fast_result!r}")
    print(f"  plain Python : {show(t_python)}")
    print(f"  PyRust Engine: {show(t_pyrust)}   (ran as: {backend})")
    print(f"  first PyRust call incl. compile: {show(first_call)}")
    if 0.9 <= speedup <= 1.1:
        print(f"  ==> about the same speed ({speedup:.2f}x)")
    elif speedup > 1:
        print(f"  ==> {speedup:.1f}x faster")
    else:
        print(f"  ==> {1 / speedup:.1f}x slower")
    RESULTS.append((name, t_python, t_pyrust, speedup, backend))
    return fast_fn

## Test 1: Integer arithmetic with branches

In [ ]:
def weighted_sum(n: int) -> int:
    total = 0
    for i in range(n):
        if i % 7 == 0:
            total += i // 3
        else:
            total += i % 11
    return total

compare("Integer arithmetic (n = 1,000,000)", weighted_sum, lambda: (1_000_000,));

## Test 2: `math` functions and a helper function

`distance` isn't decorated: PyRust compiles it together with the function that calls it.

In [ ]:
def distance(x: float, y: float) -> float:
    return math.sqrt(x * x + y * y)

def total_distance(n: int) -> float:
    total = 0.0
    for i in range(n):
        total += distance(float(i), float(n - i)) + math.sin(i) * math.log(i + 1)
    return total

compare("math + helper function (n = 300,000)", total_distance, lambda: (300_000,));

## Test 3: Recursion

Recursive functions need a return annotation (`-> int` or `-> float`).

In [ ]:
def fib(n: int) -> int:
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

compare("Recursive fib(27)", fib, lambda: (27,));

## Test 4: Bit operations (hash / checksum)

In [ ]:
def checksum(n: int) -> int:
    h = 2166136261
    for i in range(n):
        h = ((h ^ (i & 255)) * 16777619) & (2 ** 32 - 1)
        h = h ^ (h >> 13)
    return h

compare("Bit-mixing checksum (n = 500,000)", checksum, lambda: (500_000,));

## Test 5: Reading a NumPy array

In [ ]:
def sum_of_squares(values: F64Array) -> float:
    total = 0.0
    for i in range(len(values)):
        x = float(values[i])
        total += x * x
    return total

data = np.random.default_rng(1).standard_normal(500_000)
compare("Sum of squares (500,000 floats)", sum_of_squares, lambda: (data,));

## Test 6: Writing into NumPy arrays

PyRust writes into a private copy and copies it back only if the whole call succeeds, so if anything goes wrong, Python re-runs the function on the original data.

In [ ]:
def smooth(values: F64Array, out: F64Array):
    n = len(values)
    for i in range(n):
        left = float(values[i - 1]) if i > 0 else float(values[i])
        right = float(values[i + 1]) if i < n - 1 else float(values[i])
        out[i] = (left + 2.0 * float(values[i]) + right) / 4.0

compare("Smooth into an output array (500,000)", smooth, lambda: (data, np.empty_like(data)));

def histogram(values: I64Array, counts: I64Array, bins: int):
    for i in range(len(values)):
        counts[int(values[i]) % bins] += 1

labels = np.random.default_rng(2).integers(0, 1_000_000, 500_000)
compare("Histogram with int64 arrays (500,000)", histogram, lambda: (labels, np.zeros(16, dtype=np.int64), 16));

## Test 7: A tiny function

A function with no loop and no calls is too small for native code to help, so PyRust runs it as plain Python: about **1×**, never slower.

In [ ]:
def add(a, b):
    return a + b

compare("Tiny add(a, b), per call", add, lambda: (12, 30), number=10_000)
print("  dispatch:", optimize(add).report()["dispatch"]);

## Test 8: Answers never change

Rust integers are 64-bit, Python integers can be any size. When a result doesn't fit, or a math error happens, PyRust re-runs the call in Python, so you get Python's exact answer or exception.

In [ ]:
def sum_of_cubes(n: int) -> int:
    total = 0
    for i in range(n):
        total += i * i * i
    return total

fast = optimize(sum_of_cubes, warn_on_fallback=False)
for n in (1_000, 3_000_000):      # 3,000,000 overflows 64 bits -> replayed in Python
    assert fast(n) == sum_of_cubes(n)
    print(f"n = {n:>9,}: {fast(n)}  (matches plain Python)")

def roots(n: int) -> float:
    total = 0.0
    for i in range(n):
        total += math.sqrt(10 - i)   # math domain error once i > 10
    return total

fast_roots = optimize(roots, warn_on_fallback=False)
for n in (5, 20):
    try:
        print(f"roots({n}) =", fast_roots(n))
    except ValueError as exc:
        print(f"roots({n}) raised ValueError: {exc}   (same as plain Python)")

## Summary

In [ ]:
print(f"{'Test':<42}{'Python':>12}{'PyRust':>12}{'Speedup':>10}  Ran as")
print("-" * 86)
for name, t_py, t_rs, speedup, backend in RESULTS:
    print(f"{name:<42}{show(t_py):>12}{show(t_rs):>12}{speedup:>9.1f}x  {backend}")
if not RUST_OK:
    print("\nRust is not working, so everything ran as plain Python. Fix Rust (Step 1) to see the speedup.")

## What if my code can't be sped up?

PyRust tells you which line and why: once as a `PyRustFallbackWarning` when the function first runs as plain Python, and any time with `fn.explain()`.

In [ ]:
def uses_a_list(n: int) -> int:
    total = 0
    for i in range(n):
        values = [i, i + 1]          # lists aren't supported yet
        total += len(values)
    return total

slow = optimize(uses_a_list)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    slow(10)
for w in caught:
    if issubclass(w.category, PyRustFallbackWarning):
        print("Warning:", w.message)
print()
print(slow.explain())
print()
print(optimize(total_distance).explain())

In [ ]:
print(optimize(fib).rust_source()[-900:])   # the end of the Rust code PyRust generated